In [1]:
from z3 import *


In [4]:
# 1.1 Find solution + proove it is unique

x = Real("x")
equation = x**3 + 3*x**2 + 4*x + 2 == 0
s = Solver()
s.add(equation)
if s.check() == sat:
    prove(Implies(equation, x == s.model()[x]))


proved


In [6]:
# 1.2 De Morgan

p, q = Bools('p q')
prove(Or(p,q) == Not(And(Not(p), Not(q))))


proved


In [ ]:
# 2.1 Consensus theorem

p, q, r = Bools("p q r")
prove(Or(And(p, q), And(Not(p), r), And(q, r)) == Or(And(p, q), And(Not(p), r)))
prove(And(Or(p, q), Or(Not(p), r), Or(q, r)) == And(Or(p, q), Or(Not(p), r)))

# Yeah the version from claude looks a bit better, because it extracted why eqaulity holds and then derives equality from that

p, q, r = Bools("p q r")
core = Or(And(p, q), And(Not(p), r))

redundancy = Implies(And(q, r), core)        # the dropped term adds nothing
prove(redundancy)
prove(Implies(redundancy, Or(core, And(q, r)) == core))


In [ ]:
# 2.2 Clamp - proove that the resyult is in range

value, low, high = Ints("value low high")

def clamp(value, low, high):
    return If(value < low, low, If(value > high, high, value))

clamped = clamp(value, low, high)
prove(Implies(low <= high, And(low <= clamped, clamped <= high)))
prove(Implies(low <= high, clamp(clamped, low, high) == clamped))

# Claude's version was more less the same


In [ ]:
# 2.3 Page count - proove that "pages = (total + size - 1) / size" is correct for total >= 0 and size >= 0

total, size = Ints("total size")
precondition = And(total >= 0, size > 0)
pages = (total + size - 1) / size

prove(Implies(precondition, size * pages >= total))
prove(Implies(precondition, size * (pages - 1) < total))
prove(Implies(precondition, Implies(total == 0, pages == 0)))

# I prooved it pretty straighforward and did not notice that the first two claims are just the two sides
# of one range: size*(pages-1) < total <= size*pages. Only one integer can sit in that range.
# The last claim is also redundant - it is that same range at total == 0: size*(pages-1) < 0 gives pages <= 0
# and 0 <= size*pages gives pages >= 0. Claude is again smart.

prove(Implies(precondition, And(size * (pages - 1) < total, total <= size * pages)))


In [ ]:
# 2.4 Intervals overlap
# The half-open intervals [a1, a2) and [b1, b2) overlap exactly when a1 < b2 and b1 < a2. Both intervals are non empty.

a1, a2, b1, b2 = Ints("a1 a2 b1 b2")
precondition = And(a1 < a2, b1 < b2)
overlap = And(a1 < b2, b1 < a2)

prove(Implies(precondition, Not(overlap) == Or(a2 <= b1, b2 <= a1)))
prove(Implies(precondition, overlap == And(b1 < a2, a1 < b2)))

# The claude's solution is a bit smarter, because I just prooved that the idiom is consistent with itself - no overlap =
# one ends before another starts. But claude decided to proove that they overlap by finding a shared point. Smart.

a1, a2, b1, b2, point = Ints("a1 a2 b1 b2 point")
precondition = And(a1 < a2, b1 < b2)
overlap = And(a1 < b2, b1 < a2)

shared_point = Exists(point, And(a1 <= point, point < a2, b1 <= point, point < b2))
prove(Implies(precondition, overlap == shared_point))
